# Walkthrough — Kalman-fused F-K slant-stack vehicle tracking

This notebook runs the pipeline step by step on one continuous window:

1. load a channel slab from the HDF5 archive,
2. preprocess (band-pass 70–100 Hz → |strain| → median subtraction → normalisation → wavelet denoise),
3. detect vehicle crossings at a monitor channel,
4. track each vehicle with the F-K slant-stack + Kalman filter + RTS smoother.

With `USE_SYNTHETIC = True` it first writes a small synthetic archive, so it runs without the field data. To use your own data, set `USE_SYNTHETIC = False` and edit the paths and channel range.

The month-scale production run is in `scripts/run_month.py` (see the README).

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))     # run from notebooks/ without installing

%matplotlib inline
import matplotlib.pyplot as plt
import dastrack as dt
from dastrack import plotting

## 0 · Configuration

`MONITOR_CH` is relative to the loaded slab (absolute channel = `FIRST_CHANNEL + MONITOR_CH`). Every reported speed scales linearly with `dt.DX`.

In [ ]:
USE_SYNTHETIC = True

if USE_SYNTHETIC:
    from dastrack.synthetic import make_synthetic_archive
    DATA_ROOT = 'synthetic_archive'
    if not os.path.exists(DATA_ROOT):
        make_synthetic_archive(DATA_ROOT, minutes=10, n_channels=300, monitor_channel=150)
    FIRST_CHANNEL, LAST_CHANNEL, MONITOR_CH = 20, 280, 130
else:
    DATA_ROOT = '/path/to/archive'       # holds YYYY/MM/DD/HH/MM/*.h5 (UTC)
    FIRST_CHANNEL, LAST_CHANNEL = None, None   # absolute channel range of the road segment
    MONITOR_CH = 190                     # relative to the slab

DATE, HOUR, MINUTES = '2024/02/02', 0, (0, 10)
TRACKER = dt.TRACKER_DEVELOPMENT         # or dt.TRACKER_PRODUCTION (month run)
print('Detection settings:', dt.DETECTION)

## 1 · Load the window

In [ ]:
raw, fs, t0 = dt.load_window(os.path.join(DATA_ROOT, DATE, f'{HOUR:02d}'),
                             FIRST_CHANNEL, LAST_CHANNEL, *MINUTES, target_fs=dt.FS)
print(f'{raw.shape[1]} channels x {raw.shape[0] / fs:.0f} s @ {fs:g} Hz, start {t0}')

## 2 · Preprocess

The output `denoised` (time × channel, values in [0, 1]) is the shared input to detection and tracking. Vehicles appear as diagonal streaks whose slope is the inverse of their speed.

In [ ]:
filtered, norm, denoised = dt.preprocess(raw, fs, bandpass=dt.BANDPASS)
plotting.plot_preprocessing(filtered, norm, denoised, fs)
del raw, filtered, norm

## 3 · Detect crossings at the monitor channel

In [ ]:
starts, envelope = dt.detect_entries(denoised, fs, monitor_channel=MONITOR_CH,
                                     min_height=dt.DETECTION['min_height'],
                                     min_distance_sec=dt.DETECTION['min_dist_sec'])
plotting.plot_entry_detection(denoised, envelope, starts, MONITOR_CH, dt.DETECTION['min_height'], fs)

## 4 · Track one vehicle

The log shows the bootstrap F-K peaks, the seeds tried in both directions with their coherence-weighted scores, and the winner's lifecycle. The figure shows (A) the lifecycle on the waterfall, (B) the track, (C) the bootstrap F-K spectrum with the winning line *f = −vk*, (D) the slant-stack peaks, (E) the RTS speed profile with its 1-σ band and (F) a summary.

In [ ]:
r = dt.track_vehicle(starts[0], denoised, fs, dt.DX, MONITOR_CH, car_id=1, verbose=True,
                     v_min_kmh=dt.DETECTION['v_min_kmh'], v_max_kmh=dt.DETECTION['v_max_kmh'],
                     **TRACKER)
plotting.plot_vehicle_diagnostic(r, denoised, fs, dt.DX)

## 5 · Track every vehicle in the window

In [ ]:
_, results, valid = dt.run_pipeline(denoised, fs, dt.DX, MONITOR_CH, tracker_params=TRACKER,
                                    verbose=False, **dt.DETECTION)
table = dt.results_table(results)
print(f'{len(valid)} valid of {len(results)} detections')
table[table['valid']].round(2)

In [ ]:
# all valid tracks on the waterfall
m, n = denoised.shape
fig, ax = plt.subplots(figsize=(10, 7))
ax.imshow(denoised, aspect='auto', cmap='magma', vmin=0, vmax=1, extent=[0, n, m / fs, 0])
for v in valid:
    ax.plot(v['track_ch'], v['profile_t_sec'], lw=2,
            color='tab:green' if v['direction'] > 0 else 'tab:cyan')
ax.axvline(MONITOR_CH, color='yellow', ls=':')
ax.set_xlabel('Channel (relative)'); ax.set_ylabel('Time (s)')
ax.set_title(f'{len(valid)} tracked vehicles (green: +, cyan: −)')
plt.show()